In [ ]:
import pandas as pd
import numpy as np
import ast
import pickle
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('movies.csv')
df

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# Drop duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print("After dedup:", df.shape)

In [ ]:
# Fill missing values
df['Runtime(min)'].fillna(df['Runtime(min)'].median(), inplace=True)
df['Budget(million$)'].fillna(df['Budget(million$)'].median(), inplace=True)
df['Revenue(million$)'].fillna(df['Revenue(million$)'].median(), inplace=True)
df['Rating'].fillna(df['Rating'].median(), inplace=True)
df['Language'].fillna('English', inplace=True)
df['Overview'].fillna('', inplace=True)
print("Missing after fill:")
df.isnull().sum()

In [ ]:
# Convert pipe-separated strings to lists
df['Genre']    = df['Genre'].apply(lambda x: x.split('|') if isinstance(x, str) else [])
df['Cast']     = df['Cast'].apply(lambda x: x.split('|') if isinstance(x, str) else [])
df['Keywords'] = df['Keywords'].apply(lambda x: x.split('|') if isinstance(x, str) else [])
df[['Genre','Cast','Keywords']].head()

In [ ]:
# Remove spaces from multi-word tags so they become single tokens
def collapse(lst):
    return [i.replace(' ','') for i in lst]

df['Genre']    = df['Genre'].apply(collapse)
df['Cast']     = df['Cast'].apply(lambda x: collapse(x[:3]))   # top 3 cast
df['Keywords'] = df['Keywords'].apply(collapse)
df['Director'] = df['Director'].apply(lambda x: x.replace(' ',''))
df['Director'].head()

In [ ]:
# Build tags column — the feature soup
df['tags'] = (df['Genre'] + df['Cast'] + df['Keywords'] +
              df['Director'].apply(lambda x: [x]) +
              df['Overview'].apply(lambda x: x.split()))

df['tags'] = df['tags'].apply(lambda x: ' '.join(x).lower())
df[['Title','tags']].head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df['tags']).toarray()
print("Vector shape:", vectors.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)
print("Similarity matrix shape:", similarity.shape)

In [ ]:
# Test the recommendation function
def recommend(movie_title):
    if movie_title not in df['Title'].values:
        return ["Movie not found in dataset."]
    idx = df[df['Title'] == movie_title].index[0]
    distances = sorted(list(enumerate(similarity[idx])), reverse=True, key=lambda x: x[1])
    recommendations = []
    for i in distances[1:6]:
        recommendations.append(df.iloc[i[0]]['Title'])
    return recommendations

# Test it
test_movie = df['Title'].iloc[0]
print(f"Because you watched: {test_movie}")
print("You might also like:")
for r in recommend(test_movie):
    print(" -", r)

In [ ]:
# Save the model artifacts
movies_list = df[['MovieID','Title','Genre','Director','Cast','Rating','Year','Overview']].copy()
movies_list['Genre'] = movies_list['Genre'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
movies_list['Cast']  = movies_list['Cast'].apply(lambda x:  ', '.join(x) if isinstance(x, list) else x)

pickle.dump(movies_list, open('movies_list.pkl','wb'))
pickle.dump(similarity,  open('similarity.pkl','wb'))
print("✅ movies_list.pkl saved")
print("✅ similarity.pkl saved")

In [ ]:
# Verify saved files load correctly
m = pickle.load(open('movies_list.pkl','rb'))
s = pickle.load(open('similarity.pkl','rb'))
print("movies_list shape:", m.shape)
print("similarity shape :", s.shape)
m.head()